# Preprocessing & Evaluation
This notebook demonstrates the complete preprocessing pipeline:
1. Document extraction from PDF
2. Text cleaning and chunking
3. Embedding generation and vector DB creation
4. Retrieval quality evaluation

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

from src.document_loader import process_document, load_chunks
from src.embeddings import create_vector_db, get_collection, EMBEDDING_MODEL
from src.retriever import retrieve_relevant_chunks

e:\task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Document Processing
Extract text from the PDF and chunk it into overlapping segments.

In [2]:
pdf_path = project_root / 'data' / 'AI Training Document.pdf'
chunks_dir = project_root / 'chunks'
vectordb_dir = project_root / 'vectordb'

chunks = process_document(str(pdf_path), str(chunks_dir))

# Display sample chunks
print('\n--- Sample Chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {chunk['chunk_id']} (Page {chunk['source_page']}, {chunk['word_count']} words):")
    print(f"  {chunk['text'][:200]}...")

[DOC] Extracting text from: e:\task\data\AI Training Document.pdf
   Found 20 pages with text
[CHUNK] Chunking documents (larger segments for better context)...
   Created 63 chunks (including 1 summary chunk)
   Regular chunk word range: 44-265
   Average words per chunk: 205
   Summary chunk words: 369
[SAVE] Chunks saved to: e:\task\chunks\chunks.json

--- Sample Chunks:

Chunk 0 (Page 0, 369 words):
  DOCUMENT OVERVIEW: This document is titled "User Agreement".
The document spans 20 pages and covers the following main topics and sections:
Main sections include: Section 1. Introduction; Section 2. A...

Chunk 1 (Page 1, 242 words):
  User Agreement
1. Introduction
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms
posted on and in our sites, applications, tools, and services (collective...

Chunk 2 (Page 1, 247 words):
  any other country. In this User Agreement, these entities are individually and collectively referred to
as "eBay," "we,

## Step 2: Embedding Generation & Vector DB Creation
Generate embeddings for all chunks and store them in ChromaDB.

In [3]:
print(f'Embedding Model: {EMBEDDING_MODEL}')

collection = create_vector_db(chunks, str(vectordb_dir))
print(f'\n--- Collection stats:')
print(f'   Total documents: {collection.count()}')

Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9669.70it/s]


   Indexed chunks 1-50 of 63
   Indexed chunks 51-63 of 63
[DONE] Vector DB created with 63 documents

--- Collection stats:
   Total documents: 63


## Step 3: Retrieval Quality Evaluation
Test the retrieval with sample queries to ensure relevant chunks are pulled.

In [4]:
test_queries = [
    "Which specific eBay entity am I contracting with if I reside in the United Kingdom?",
    "Under the Agreement to Arbitrate, what is the mandatory precondition before commencing arbitration, and exactly how long does it last?",
    "How do I change the oil and replace the filters on a 2024 Ford F-150?",
    "The document says eBay guarantees 100% accuracy for its AI tools. Can you explain how they achieve this?",
    "I am a 16-year-old living in the United States. Which eBay entity do I contract with?",
    "Can you explain the specific $50 flat fee for selling a used smartphone mentioned in Section 6?",
    "Write a 500-word romantic poem about the Agreement to Arbitrate in Section 19.",
    "In your opinion, is the 'Agreement to Arbitrate' fair for the average consumer, or is it designed just to protect eBay's profits?",
]

for query in test_queries:
    print(f'\n[QUERY] {query}')
    print('-' * 50)
    
    results = retrieve_relevant_chunks(query, top_k=3, persist_dir=str(vectordb_dir))
    
    for i, result in enumerate(results, 1):
        print(f"  {i}. [Score: {result['relevance_score']:.4f}] Page {result['source_page']} | {result['text'][:120]}...")



[QUERY] Which specific eBay entity am I contracting with if I reside in the United Kingdom?
--------------------------------------------------
  1. [Score: 0.7992] Page 1 | User Agreement
1. Introduction
This User Agreement, the Mobile Application Terms of Use, and all policies and additional...
  2. [Score: 0.4541] Page 1 | any other country. In this User Agreement, these entities are individually and collectively referred to
as "eBay," "we,"...
  3. [Score: 0.4350] Page 11 | Payments for goods and services sold using our Services are facilitated by designated eBay entities
(each, an "eBay Paym...

[QUERY] Under the Agreement to Arbitrate, what is the mandatory precondition before commencing arbitration, and exactly how long does it last?
--------------------------------------------------
  1. [Score: 0.7559] Page 14 | The Informal Dispute Resolution process lasts 45 days and is a mandatory precondition to
commencing arbitration. The sta...
  2. [Score: 0.5833] Page 14 | necessary to